In [1]:
# ============================================================
# 027_human_correction_ingest_to_notion
# ============================================================
#
# Overview
# ----------------
# End-to-end pipeline to convert pasted LLM chat logs
# (original user intent + iterative debugging / correction conversation)
# into structured, Skill-ready "human correction" records.
#
# High-value corrections are extracted via OpenAI API, normalized into a
# reusable schema, deduplicated via JSONL, and persisted to Notion using
# a schema-adaptive REST-based writer.
#
# The resulting records are designed to be directly reusable as Skills
# for future LLM / Agent systems.
#
# Inputs / Outputs
# ----------------
# Input:
#   - Local text file containing:
#       * Original user prompt (optional)
#       * LLM / human debugging or revision conversation
#
# Output:
#   - New pages in Notion database (NOTION_HC_DB_ID)
#     with Skill-ready, schema-adaptive properties
#   - Local JSONL artifact (skills/human_corrections.jsonl)
#     as the source of truth for deduplication and offline inspection
#
# Structure
# ----------------
# Cell 01: Imports and global configuration
# Cell 02: Environment loading and client initialization
# Cell 03: Notion authentication validation (REST-based)
# Cell 04: Notion database schema introspection
# Cell 05: Input file loader and preprocessing
# Cell 06: OpenAI extraction (high-value human corrections, JSON-only)
# Cell 07: Normalization to Skill-ready records (IDs, timestamps, metadata)
# Cell 08: Local JSONL artifact writer (hash-based deduplication)
# Cell 09: Notion database page writer (REST-based, schema-adaptive)
# Cell 10: End-to-end pipeline orchestrator
# Cell 11: Example invocation and interactive UI (file picker)
#
# Notes
# ----------------
# - Explicitly uses load_dotenv("env.txt") (not default .env)
# - Extracts ONLY high-value corrections:
#     reasoning, scope, method, constraint, or structural changes
# - DRY_RUN supported (default True) for safe validation
# - JSONL is the source of truth; Notion is a synchronized projection
# - Notion writes use REST API to avoid SDK-version instability
# - Date / created_time properties are handled explicitly (Created At supported)
# - Handles partial failures gracefully and continues processing
# - Idempotent ingestion via hash-based deduplication
# - Token-safe truncation for long chat logs


In [2]:
# ============================================================
# Cell 01 — Imports and global configuration
# ============================================================
# Overview:
#   Core imports and global constants for the human correction ingestion pipeline.
#   Loads environment from env.txt, configures OpenAI client parameters,
#   and sets global flags for dry-run mode and batch processing limits.
#
# Inputs / Outputs:
#   None directly; sets up shared runtime state.
#
# Notes:
#   - DRY_RUN defaults to True to prevent accidental writes during testing.
#   - MAX_RECORDS caps processing for large input files.
#   - Uses explicit env.txt (not default .env) per project conventions.
#

# --- Mandatory env loading ---
from dotenv import load_dotenv
load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Standard library imports ---
import os
import sys
import json
import hashlib
import re
from pathlib import Path
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional

# --- Third-party imports ---
import openai
from notion_client import Client as NotionClient

# --- Global configuration ---
DRY_RUN = True  # Set False to enable actual Notion writes
MAX_RECORDS = None  # None for unlimited, or int to cap processing
OUTPUT_JSONL_PATH = Path('skills/human_corrections.jsonl')
INPUT_FOLDER = Path('data/chat_logs')  # Expected location for input text files

# --- Notion database IDs (read from environment) ---
# Expected env vars: NOTION_TOKEN, NOTION_HC_DB_ID
# Schema columns: Title (title), Original_Prompt (rich_text), Correction_Type (select),
#                 Old_Approach (rich_text), New_Approach (rich_text), Reasoning (rich_text),
#                 Source (select), Timestamp (date)

# --- OpenAI configuration ---
# Expected env var: OPENAI_API_KEY (loaded by dotenv)

# --- Logging helpers ---
def log_info(msg: str) -> None:
    """Safe logging that never prints secrets."""
    print(f"[INFO] {msg}")

def log_error(msg: str) -> None:
    """Safe error logging."""
    print(f"[ERROR] {msg}", file=sys.stderr)

log_info(f"Configuration loaded: DRY_RUN={DRY_RUN}, MAX_RECORDS={MAX_RECORDS}")
log_info(f"LLM: {llm_provider}/{llm_model} @ temp={llm_temperature}")


[INFO] Configuration loaded: DRY_RUN=True, MAX_RECORDS=None
[INFO] LLM: OpenAI/gpt-4o-mini @ temp=0.0


In [3]:
# ============================================================
# Cell 02 — Environment loading and client initialization
# ============================================================
# Overview:
#   Loads secrets from env.txt and initializes OpenAI and Notion clients.
#   Validates that all required environment variables are present.
#   Never logs or prints secrets; raises clear errors if configuration is missing.
#
# Inputs / Outputs:
#   Reads: OPENAI_API_KEY, NOTION_TOKEN, NOTION_HC_DB_ID from environment.
#   Creates: openai_client (OpenAI), notion_client (NotionClient), notion_hc_db_id (str).
#
# Notes:
#   - env.txt must be present and contain all required keys.
#   - Clients are initialized but not tested here (see Cell 03 for validation).
#   - Raises ValueError immediately if any secret is missing.
#

# --- Load environment variables ---
# Already called in Cell 01, but ensure it's effective
load_dotenv('env.txt', override=False)

# --- Validate required secrets ---
required_env_vars = {
    'OPENAI_API_KEY': 'OpenAI API authentication',
    'NOTION_TOKEN': 'Notion integration token',
    'NOTION_HC_DB_ID': 'Notion human corrections database ID'
}

missing_vars = []
for var_name, description in required_env_vars.items():
    if not os.getenv(var_name):
        missing_vars.append(f"{var_name} ({description})")

if missing_vars:
    error_msg = "Missing required environment variables:\n" + "\n".join(f"  - {v}" for v in missing_vars)
    log_error(error_msg)
    raise ValueError(error_msg)

log_info("All required environment variables found")

# --- Initialize OpenAI client ---
try:
    openai_client = openai.OpenAI(
        api_key=os.getenv('OPENAI_API_KEY')
    )
    log_info(f"OpenAI client initialized (model: {llm_model})")
except Exception as e:
    log_error(f"Failed to initialize OpenAI client: {e}")
    raise

# --- Initialize Notion client ---
try:
    notion_client = NotionClient(
        auth=os.getenv('NOTION_TOKEN')
    )
    notion_hc_db_id = os.getenv('NOTION_HC_DB_ID')
    log_info("Notion client initialized")
    log_info(f"Target database ID: {notion_hc_db_id[:8]}...{notion_hc_db_id[-4:]}")
except Exception as e:
    log_error(f"Failed to initialize Notion client: {e}")
    raise

# --- Create output directory if needed ---
OUTPUT_JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)
log_info(f"Output directory ready: {OUTPUT_JSONL_PATH.parent}")

log_info("Client initialization complete")


[INFO] All required environment variables found
[INFO] OpenAI client initialized (model: gpt-4o-mini)
[INFO] Notion client initialized
[INFO] Target database ID: 2ee8e0e4...6cdf
[INFO] Output directory ready: skills
[INFO] Client initialization complete


In [4]:
# ============================================================
# Cell 03 — Notion authentication validation (REST-based, self-contained)
# ============================================================
# Overview:
#   Validates Notion authentication and database access using Notion REST API.
#   This cell is self-contained: it (re)builds NOTION_HEADERS from env to
#   avoid execution-order issues.
#
# Inputs / Outputs:
#   Reads: env vars (NOTION_TOKEN, NOTION_VERSION, NOTION_HC_DB_ID)
#   Outputs: Logs validation status; raises on failure
#
# Notes:
#   - Read-only checks:
#       1) GET /v1/users/me (auth)
#       2) GET /v1/databases/{db_id} (DB access)
#       3) POST /v1/databases/{db_id}/query (query permission)
#   - Never prints secrets
#

import os, requests

def _mask_id(s: str) -> str:
    if not s:
        return ""
    return f"{s[:8]}...{s[-4:]}" if len(s) > 12 else s

NOTION_TOKEN   = os.getenv("NOTION_TOKEN")
NOTION_VERSION = os.getenv("NOTION_VERSION")
NOTION_HC_DB_ID = os.getenv("NOTION_HC_DB_ID", "").strip()

if not NOTION_TOKEN:
    raise ValueError("NOTION_TOKEN is missing.")
if not NOTION_VERSION:
    raise ValueError("NOTION_VERSION is missing.")
if not NOTION_HC_DB_ID:
    raise ValueError("NOTION_HC_DB_ID is missing.")

NOTION_HEADERS = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

def notion_get_me():
    return requests.get("https://api.notion.com/v1/users/me", headers=NOTION_HEADERS, timeout=30)

def notion_get_database(db_id: str):
    return requests.get(f"https://api.notion.com/v1/databases/{db_id}", headers=NOTION_HEADERS, timeout=30)

def notion_query_data_source(data_source_id: str, page_size: int = 1):
    url = f"https://api.notion.com/v1/data_sources/{data_source_id}/query"
    print("QUERY URL =", url)
    return requests.post(url, headers=NOTION_HEADERS, json={"page_size": page_size}, timeout=30)

# ---- run checks ----
log_info("[INFO] Validating Notion authentication...")

r = notion_get_me()
if r.status_code != 200:
    raise ValueError(f"Auth check failed: HTTP {r.status_code} - {r.text}")
log_info("[INFO] ✅ Notion token is valid (users/me OK)")

log_info(f"[INFO] Checking database access: {_mask_id(NOTION_HC_DB_ID)}")
rdb = notion_get_database(NOTION_HC_DB_ID)
if rdb.status_code != 200:
    raise ValueError(f"Database retrieve failed: HTTP {rdb.status_code} - {rdb.text}")
log_info("[INFO] ✅ Database retrieve OK (integration can access the DB)")

db = rdb.json()
data_sources = db.get("data_sources") or []
if not data_sources:
    raise ValueError(f"No data_sources found in database response. Keys={list(db.keys())}")

data_source_id = data_sources[0]["id"]
log_info(f"[INFO] Using data_source_id: {_mask_id(data_source_id)}")

rq = notion_query_data_source(data_source_id, page_size=1)
if rq.status_code != 200:
    raise ValueError(f"Data source query failed: HTTP {rq.status_code} - {rq.text}")

data = rq.json()
log_info(f"[INFO] ✅ Data source query OK (sample results: {len(data.get('results', []))})")
log_info("[INFO] Notion validation complete ✅")


[INFO] [INFO] Validating Notion authentication...
[INFO] [INFO] ✅ Notion token is valid (users/me OK)
[INFO] [INFO] Checking database access: 2ee8e0e4...6cdf
[INFO] [INFO] ✅ Database retrieve OK (integration can access the DB)
[INFO] [INFO] Using data_source_id: 2ee8e0e4...89bb
QUERY URL = https://api.notion.com/v1/data_sources/2ee8e0e4-d162-809d-b0a7-000b18f389bb/query
[INFO] [INFO] ✅ Data source query OK (sample results: 1)
[INFO] [INFO] Notion validation complete ✅


In [5]:
# ============================================================
# Cell 04 — Notion database schema introspection (REST-based)
# ============================================================
# Overview:
#   Retrieves the schema of the target Notion database via REST API and
#   builds a property map for safe writes (auto-adapts to your DB schema).
#
# Inputs / Outputs:
#   Reads: NOTION_HEADERS, NOTION_HC_DB_ID
#   Outputs:
#     - notion_db_info (dict)
#     - notion_schema (dict)
#     - notion_property_map (dict): {prop_name: {"type": ..., "id": ...}}
#     - notion_title_prop (str): the name of the title property
#
# Notes:
#   - Uses GET /v1/databases/{db_id}
#   - Does NOT enforce a hardcoded schema (no more "missing required" failures)
#   - Later cells should use notion_property_map + notion_title_prop to write safely
#

import requests

def notion_retrieve_database(db_id: str):
    return requests.get(
        f"https://api.notion.com/v1/databases/{db_id}",
        headers=NOTION_HEADERS,
        timeout=30
    )

def notion_retrieve_data_source(data_source_id: str):
    return requests.get(
        f"https://api.notion.com/v1/data_sources/{data_source_id}",
        headers=NOTION_HEADERS,
        timeout=30
    )

log_info("[INFO] Retrieving Notion database (container) ...")

rdb = notion_retrieve_database(NOTION_HC_DB_ID)
if rdb.status_code != 200:
    raise ValueError(f"Failed to retrieve database: HTTP {rdb.status_code} - {rdb.text}")

db_obj = rdb.json()

# DB title for logging
title_arr = db_obj.get("title", [])
db_title = "".join([t.get("plain_text", "") for t in title_arr]) if title_arr else "Untitled"
log_info(f"[INFO] Database title: {db_title}")

data_sources = db_obj.get("data_sources") or []
if not data_sources:
    raise ValueError(f"No data_sources found in database object. Keys={list(db_obj.keys())}")

data_source_id = data_sources[0]["id"]  # 通常は1つ
log_info(f"[INFO] Using data_source_id: {data_source_id[:8]}...{data_source_id[-4:]}")

log_info("[INFO] Retrieving Notion data source schema (REST)...")
rds = notion_retrieve_data_source(data_source_id)
if rds.status_code != 200:
    raise ValueError(f"Failed to retrieve data source: HTTP {rds.status_code} - {rds.text}")

notion_ds_info = rds.json()
notion_schema = notion_ds_info.get("properties", {}) or {}

log_info(f"[INFO] Retrieved {len(notion_schema)} properties (from data source)")

# Build property map + find title prop
notion_property_map = {}
notion_title_prop = None

for prop_name, prop_cfg in notion_schema.items():
    ptype = prop_cfg.get("type")
    pid = prop_cfg.get("id")
    notion_property_map[prop_name] = {"type": ptype, "id": pid, "config": prop_cfg}
    if ptype == "title":
        notion_title_prop = prop_name

if not notion_title_prop:
    raise ValueError("No title property found in the data source schema.")

log_info(f"[INFO] Title property detected: '{notion_title_prop}'")
log_info("[INFO] Schema introspection complete ✅")

log_info("[INFO] Properties (name:type):")
for k, v in list(notion_property_map.items())[:50]:
    log_info(f"  - {k}: {v['type']}")



[INFO] [INFO] Retrieving Notion database (container) ...
[INFO] [INFO] Database title: HC_DB
[INFO] [INFO] Using data_source_id: 2ee8e0e4...89bb
[INFO] [INFO] Retrieving Notion data source schema (REST)...
[INFO] [INFO] Retrieved 9 properties (from data source)
[INFO] [INFO] Title property detected: 'Skill ID'
[INFO] [INFO] Schema introspection complete ✅
[INFO] [INFO] Properties (name:type):
[INFO]   - Created At: date
[INFO]   - Human Correction: rich_text
[INFO]   - Cell Tag: rich_text
[INFO]   - Input: rich_text
[INFO]   - Status: select
[INFO]   - Model Output: rich_text
[INFO]   - Notebook: rich_text
[INFO]   - Reason Tags: multi_select
[INFO]   - Skill ID: title


In [6]:
# ============================================================
# Cell 05 — Input file loader and preprocessing
# ============================================================
# Overview:
#   Loads chat log text files from disk and preprocesses them for extraction.
#   Splits content into original prompt and conversation segments when possible.
#   Never hard-fails on format: if splitting is not possible, treats everything
#   as conversation-only and continues.
#
# Inputs / Outputs:
#   Input: Path to a .txt file containing (optionally) original prompt + conversation
#   Output: Dict with at least:
#     - source_file (str)
#     - source_path (str)
#     - original_prompt (str)
#     - conversation (str)
#     - raw_text (str)
#
# Notes:
#   - Accepts many formats: explicit headers, role markers, or plain text
#   - Designed to be resilient to “copy-paste” chat logs
#

from pathlib import Path
from typing import List, Dict
import re


def load_chat_log(file_path: Path) -> Dict[str, str]:
    """
    Robust chat log loader.

    Tries to split into:
      - original_prompt
      - conversation

    Never fails due to formatting: if split is not possible, treats everything
    as conversation-only.

    Returns:
      {
        "source_file": str,    # filename only
        "source_path": str,    # full path
        "file_path": str,      # backward-compat alias
        "original_prompt": str,
        "conversation": str,
        "raw_text": str
      }
    """
    log_info(f"Loading chat log from: {file_path.name}")

    text = Path(file_path).read_text(encoding="utf-8", errors="replace").strip()
    if not text:
        raise ValueError(f"Empty file: {file_path.name}")

    base = {
        "source_file": file_path.name,       # ✅ required by downstream code
        "source_path": str(file_path),
        "file_path": str(file_path),         # compatibility
        "raw_text": text,
    }

    # --- 1) Try explicit section headers ---
    # We accept patterns like:
    #   Original Prompt:
    #   Conversation:
    # or just Conversation: as separator (prompt = before)
    # Use flexible matching and case-insensitive
    # Strategy:
    #   - if both headers exist: split by them
    #   - else if Conversation exists: split at Conversation
    #   - else: continue to other heuristics

    # a) Both "Original Prompt:" and "Conversation:" present
    m_op = re.search(r"^\s*Original\s*Prompt\s*:\s*$", text, flags=re.IGNORECASE | re.MULTILINE)
    m_cv = re.search(r"^\s*Conversation\s*:\s*$", text, flags=re.IGNORECASE | re.MULTILINE)
    if m_op and m_cv and m_cv.start() > m_op.end():
        op_text = text[m_op.end():m_cv.start()].strip()
        cv_text = text[m_cv.end():].strip()
        if cv_text:
            return {
                **base,
                "original_prompt": op_text,
                "conversation": cv_text,
            }

    # b) Conversation header only (prompt = before it)
    m_cv2 = re.search(r"^\s*Conversation\s*:\s*$", text, flags=re.IGNORECASE | re.MULTILINE)
    if m_cv2:
        before = text[:m_cv2.start()].strip()
        after = text[m_cv2.end():].strip()
        if after:
            return {
                **base,
                "original_prompt": before,
                "conversation": after,
            }

    # --- 2) Try bracketed role markers: [USER], [ASSISTANT] ---
    # Treat the first [USER] block as original prompt and everything after first [ASSISTANT] as conversation
    if re.search(r"^\s*\[USER\]\s*$", text, flags=re.MULTILINE):
        user_match = re.search(r"^\s*\[USER\]\s*$", text, flags=re.MULTILINE)
        if user_match:
            rest = text[user_match.end():].lstrip()
            a_match = re.search(r"^\s*\[ASSISTANT\]\s*$", rest, flags=re.MULTILINE)
            if a_match:
                original_prompt = rest[:a_match.start()].strip()
                conversation = rest[a_match.start():].strip()
                if conversation:
                    return {
                        **base,
                        "original_prompt": original_prompt,
                        "conversation": conversation,
                    }

    # --- 3) Try common "User:" / "Assistant:" patterns ---
    # Take everything until first Assistant as original prompt; rest as conversation
    if re.search(r"^\s*(User|USER)\s*:\s*", text, flags=re.MULTILINE) and \
       re.search(r"^\s*(Assistant|ASSISTANT)\s*:\s*", text, flags=re.MULTILINE):
        first_assistant = re.search(r"^\s*(Assistant|ASSISTANT)\s*:\s*", text, flags=re.MULTILINE)
        if first_assistant:
            original_prompt = text[:first_assistant.start()].strip()
            conversation = text[first_assistant.start():].strip()
            if conversation:
                return {
                    **base,
                    "original_prompt": original_prompt,
                    "conversation": conversation,
                }

    # --- 4) Fallback: no split possible ---
    log_info(
        "Could not reliably split into prompt/conversation; "
        "falling back to conversation-only mode."
    )
    return {
        **base,
        "original_prompt": "",
        "conversation": text,
    }


def discover_chat_logs(folder: Path, pattern: str = "*.txt") -> List[Path]:
    """
    Find all chat log files in a folder.

    Args:
        folder: Directory to search
        pattern: Glob pattern for file matching

    Returns:
        List of Path objects sorted by modification time (oldest first)
    """
    if not folder.exists():
        log_error(f"Input folder does not exist: {folder}")
        return []

    files = sorted(folder.glob(pattern), key=lambda p: p.stat().st_mtime)
    log_info(f"Found {len(files)} chat log file(s) in {folder}")
    return files


# --- Test loader on first available file (if INPUT_FOLDER is defined) ---
try:
    _input_folder = INPUT_FOLDER  # may not exist depending on notebook execution order
except NameError:
    _input_folder = None

if _input_folder is not None:
    if _input_folder.exists():
        test_files = discover_chat_logs(_input_folder)
        if test_files:
            try:
                test_log = load_chat_log(test_files[0])
                log_info(f"Test load successful: {test_log.get('source_file', '(unknown)')}")
                log_info(f"  Prompt preview: {(test_log.get('original_prompt','')[:80] + '...') if test_log.get('original_prompt') else '(empty)'}")
                log_info(f"  Conversation preview: {(test_log.get('conversation','')[:80] + '...') if test_log.get('conversation') else '(empty)'}")
            except Exception as e:
                log_error(f"Test load failed: {e}")
        else:
            log_info(f"No test files found in {str(_input_folder)}")
    else:
        log_info(f"Input folder not yet created: {str(_input_folder)}")

log_info("Input loader ready ✅")


[INFO] Found 0 chat log file(s) in data/chat_logs
[INFO] No test files found in data/chat_logs
[INFO] Input loader ready ✅


In [7]:
# ============================================================
# Cell 06 — OpenAI extraction function (Skill/Notion-ready schema)
# ============================================================
# Overview:
#   Calls OpenAI API to extract Skill-ready records from chat logs.
#   Outputs objects that map directly to your Notion DB columns:
#     - title, skill_id, input, human_correction, model_output, notebook,
#       reason_tags, status, created_at, cell_tag, source_file, context_snippet
#
# Inputs / Outputs:
#   Input: Dict from load_chat_log (original_prompt, conversation, source_file, raw_text)
#   Output: List[dict] of Skill-ready records (validated)
#
# Notes:
#   - Uses response_format={"type":"json_object"} and strict json.loads validation
#   - Robust to missing/empty original_prompt (conversation-only mode)
#   - Basic token-budget truncation to reduce context overflow
#   - Designed so Cell 07 can be simplified to "add IDs + truncation"
#

import time
import json
from typing import List, Dict, Any, Optional
from pathlib import Path

# ---- Defaults (safe fallbacks if globals not defined elsewhere) ----
DEFAULT_LLM_MODEL = "gpt-4o-mini"
DEFAULT_LLM_TEMPERATURE = 0.2

try:
    llm_model
except NameError:
    llm_model = DEFAULT_LLM_MODEL

try:
    llm_temperature
except NameError:
    llm_temperature = DEFAULT_LLM_TEMPERATURE

# Optional global metadata you can set in Cell 01 (recommended)
# Example:
# NOTEBOOK_NAME = "027_hc_ingest.ipynb"
# CELL_TAG = "Cell 06"
try:
    NOTEBOOK_NAME
except NameError:
    NOTEBOOK_NAME = ""

try:
    CELL_TAG
except NameError:
    CELL_TAG = "Cell 06"

try:
    SKILL_ID
except NameError:
    # Matches what we used in Cell 07 normalization fallback
    SKILL_ID = "human_correction_extract"


# ---- Extraction Prompt (Notion-ready) ----
EXTRACTION_SYSTEM_PROMPT = """You are an expert at extracting reusable "skills" from debugging conversations.

You will be given:
- an original user prompt (may be empty)
- a debugging conversation log (may be long)

Your goal:
Extract ONLY high-value, reusable corrections that can teach an agent how to do better next time.

A "high-value correction" must include at least one of:
- methodological change (approach/architecture)
- scope/constraint clarification
- error-handling/robustness improvement
- interface / contract clarification
- non-trivial edge case handling

DO NOT extract:
- typos, wording tweaks, code style nits
- purely mechanical fixes with no transferable lesson

Return JSON ONLY in this format:

{
  "records": [
    {
      "title": "Short title (max 80 chars)",
      "input": "Context / situation in which this skill applies (1-3 sentences)",
      "human_correction": {
        "before": "What the model/system did before (concise)",
        "after": "What should be done instead (concise)",
        "note": "Optional: brief rule-of-thumb for humans/agents"
      },
      "model_output": "Optional: what the model initially produced that was flawed (brief)",
      "reason_tags": ["tag1","tag2","tag3"],
      "cell_tag": "Optional: e.g., Cell 06 / Validation / Loader",
      "status": "raw"
    }
  ]
}

Guidelines:
- reason_tags must be 1-6 short lowercase snake_case tags, e.g.
  ["missing_scope","sdk_dependency","robustness","format_mismatch","deduplication","rate_limit"]
- Prefer concrete before/after; include key API endpoints or function names when relevant.
- If nothing qualifies, return {"records": []}.
"""

# ---- Simple token-budget guard (character-based truncation) ----
MAX_CONV_CHARS = 35_000
HEAD_CHARS = 18_000
TAIL_CHARS = 17_000

def _truncate_conversation(text: str) -> str:
    text = text or ""
    if len(text) <= MAX_CONV_CHARS:
        return text
    head = text[:HEAD_CHARS]
    tail = text[-TAIL_CHARS:]
    return head + "\n\n...[TRUNCATED]...\n\n" + tail

def build_extraction_prompt(chat_log: Dict[str, Any]) -> str:
    """Build user prompt for extraction (robust to missing keys)."""
    original_prompt = (chat_log.get("original_prompt") or "").strip()
    conversation = (chat_log.get("conversation") or "").strip()
    if not conversation:
        conversation = (chat_log.get("raw_text") or "").strip()

    conversation = _truncate_conversation(conversation)

    return f"""Original Prompt (may be empty):
{original_prompt}

---

Debugging Conversation:
{conversation}

---

Extract Skill-ready records as JSON ONLY (see required schema)."""

def _safe_json_loads(s: str) -> Dict[str, Any]:
    if not s or not isinstance(s, str):
        raise ValueError("Empty response content (expected JSON string)")
    return json.loads(s)

def _to_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    try:
        return str(x)
    except Exception:
        return ""

def _cap(s: str, n: int) -> str:
    s = _to_str(s).strip()
    return s[:n] if len(s) > n else s

def _normalize_tags(tags: Any) -> List[str]:
    """
    Normalize reason_tags to a list of short snake_case strings.
    """
    if tags is None:
        return []
    if isinstance(tags, str):
        tags = [t.strip() for t in tags.split(",") if t.strip()]
    if not isinstance(tags, list):
        return []
    cleaned = []
    for t in tags:
        t = _to_str(t).strip().lower()
        if not t:
            continue
        # very light normalization
        t = t.replace(" ", "_").replace("-", "_")
        if len(t) > 40:
            t = t[:40]
        cleaned.append(t)
    # unique, keep order
    uniq = []
    seen = set()
    for t in cleaned:
        if t not in seen:
            uniq.append(t)
            seen.add(t)
    return uniq[:6]


def extract_corrections_via_openai(
    chat_log: Dict[str, Any],
    max_retries: int = 3,
    timeout: int = 60
) -> List[Dict[str, Any]]:
    """
    Extract Skill-ready records from chat log using OpenAI API.

    Returns:
        List of validated dicts, each includes keys compatible with Notion DB:
          - title, skill_id, input, human_correction, model_output, notebook,
            reason_tags, status, created_at, cell_tag, source_file, context_snippet
    """
    source_file = chat_log.get("source_file") or Path(chat_log.get("file_path", "unknown.txt")).name
    log_info(f"Extracting corrections from: {source_file}")

    user_prompt = build_extraction_prompt(chat_log)

    for attempt in range(1, max_retries + 1):
        try:
            log_info(f"API call attempt {attempt}/{max_retries}...")

            response = openai_client.chat.completions.create(
                model=llm_model,
                temperature=llm_temperature,
                messages=[
                    {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format={"type": "json_object"},
                timeout=timeout,
            )

            usage = getattr(response, "usage", None)
            if usage:
                log_info(
                    f"Token usage: {usage.prompt_tokens} prompt + "
                    f"{usage.completion_tokens} completion = {usage.total_tokens} total"
                )

            content = response.choices[0].message.content
            result = _safe_json_loads(content)

            if not isinstance(result, dict):
                raise ValueError("Response JSON is not an object")

            records = result.get("records", [])
            if records is None:
                records = []
            if not isinstance(records, list):
                raise ValueError("'records' field is not a list")

            validated: List[Dict[str, Any]] = []

            # Build a short context snippet (useful for debugging)
            conv = (chat_log.get("conversation") or chat_log.get("raw_text") or "")
            conv = _to_str(conv).strip()
            context_snippet_default = _cap(conv, 200)

            for i, r in enumerate(records, 1):
                if not isinstance(r, dict):
                    log_info(f"Skipping record {i}: not an object")
                    continue

                title = _cap(r.get("title", ""), 80)
                input_text = _cap(r.get("input", ""), 2000)

                hc = r.get("human_correction", {}) or {}
                if not isinstance(hc, dict):
                    hc = {}

                before = _cap(hc.get("before", ""), 2000)
                after  = _cap(hc.get("after", ""), 2000)
                note   = _cap(hc.get("note", ""), 1000)

                model_output = _cap(r.get("model_output", ""), 2000)
                reason_tags = _normalize_tags(r.get("reason_tags", []))
                status = _cap(r.get("status", "raw") or "raw", 40).lower()
                cell_tag = _cap(r.get("cell_tag", CELL_TAG), 200)

                # Minimal quality gates:
                # - title must exist
                # - (before & after) should exist OR (input & after) should exist
                if not title:
                    log_info(f"Skipping record {i}: empty title")
                    continue
                if not after and not (before and after):
                    log_info(f"Skipping record {i}: missing 'after' in human_correction")
                    continue

                item = {
                    # Core Notion-facing fields
                    "title": title,
                    "skill_id": SKILL_ID,
                    "input": input_text,
                    "human_correction": {
                        "before": before,
                        "after": after,
                        "note": note
                    },
                    "model_output": model_output,

                    # Metadata that later cells can fill more (Cell 07)
                    "notebook": NOTEBOOK_NAME,
                    "reason_tags": reason_tags,
                    "status": status or "raw",
                    "created_at": "",  # filled in Cell 07 (UTC ISO)
                    "cell_tag": cell_tag,

                    # Traceability
                    "source_file": source_file,
                    "context_snippet": _cap(r.get("context_snippet", context_snippet_default), 200),
                }

                validated.append(item)

            log_info(f"✓ Extracted {len(validated)} valid record(s)")
            return validated

        except json.JSONDecodeError as e:
            log_error(f"Attempt {attempt} failed: Invalid JSON from API: {e}")
            if attempt == max_retries:
                raise ValueError(f"OpenAI returned invalid JSON after {max_retries} attempts") from e

        except Exception as e:
            log_error(f"Attempt {attempt} failed: {type(e).__name__}: {e}")
            if attempt == max_retries:
                raise

        if attempt < max_retries:
            wait_time = 2 ** attempt
            log_info(f"Retrying in {wait_time}s...")
            time.sleep(wait_time)

    return []


log_info("OpenAI extraction (Skill/Notion-ready) ready ✅")


[INFO] OpenAI extraction (Skill/Notion-ready) ready ✅


In [8]:
# ============================================================
# Cell 07 — Normalization to Skill-ready records (v2: records schema)
# ============================================================
# Overview:
#   Normalizes Skill/Notion-ready "records" extracted in Cell 06.
#   Adds:
#     - stable id (hash) for deduplication
#     - created_at timestamp (UTC ISO8601)
#   Enforces length limits for Notion-friendly fields.
#   Produces both structured and text-friendly forms:
#     - human_correction (dict) + human_correction_text (rich_text-friendly)
#     - reason_tags (list) + reason_tags_text (fallback)
#
# Inputs / Outputs:
#   Input: List[dict] from extract_corrections_via_openai (Cell 06)
#   Output: List[dict] normalized for JSONL + Notion
#
# Notes:
#   - Notion rich_text practical limit is ~2000 chars per block; we cap to 2000.
#   - Title is capped to 80 (as per Cell 06), but Notion title itself can be longer.
#   - Keeps source_file + context_snippet for traceability.
#

from datetime import datetime, timezone
import hashlib
from typing import List, Dict, Any

# --- Safe defaults (avoid NameError due to execution order) ---
try:
    NOTEBOOK_NAME
except NameError:
    NOTEBOOK_NAME = ""

try:
    CELL_TAG
except NameError:
    CELL_TAG = "Cell 07"

try:
    SKILL_ID
except NameError:
    SKILL_ID = "human_correction_extract"


TEXT_MAX = 2000      # rich_text safety cap
TITLE_MAX = 2000     # Notion title field cap (we keep safe)
SNIPPET_MAX = 200
NOTE_MAX = 1000


def _s(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    try:
        return str(x)
    except Exception:
        return ""


def _cap(s: str, n: int) -> str:
    s = _s(s).strip()
    return s[:n] if len(s) > n else s


def _normalize_tags(tags: Any) -> List[str]:
    """
    Normalize reason_tags to a list of unique lowercase snake_case tokens.
    """
    if tags is None:
        return []
    if isinstance(tags, str):
        tags = [t.strip() for t in tags.split(",") if t.strip()]
    if not isinstance(tags, list):
        return []
    cleaned = []
    for t in tags:
        t = _s(t).strip().lower()
        if not t:
            continue
        t = t.replace(" ", "_").replace("-", "_")
        if len(t) > 40:
            t = t[:40]
        cleaned.append(t)
    # unique preserve order
    out = []
    seen = set()
    for t in cleaned:
        if t not in seen:
            out.append(t)
            seen.add(t)
    return out[:6]


def _make_hc_text(hc: Dict[str, Any]) -> str:
    """
    Build a Notion-friendly text representation of human_correction.
    """
    before = _cap(hc.get("before", ""), TEXT_MAX)
    after = _cap(hc.get("after", ""), TEXT_MAX)
    note = _cap(hc.get("note", ""), NOTE_MAX)

    parts = []
    if before:
        parts.append("BEFORE:\n" + before)
    if after:
        parts.append("AFTER:\n" + after)
    if note:
        parts.append("NOTE:\n" + note)

    return _cap("\n\n".join(parts), TEXT_MAX)


def _hash_id(record: Dict[str, Any]) -> str:
    """
    Stable-ish ID based on key semantic fields.
    """
    title = _s(record.get("title", "")).strip()
    input_text = _s(record.get("input", "")).strip()
    hc = record.get("human_correction", {}) or {}
    if not isinstance(hc, dict):
        hc = {}
    before = _s(hc.get("before", "")).strip()
    after = _s(hc.get("after", "")).strip()

    material = f"{title}|{input_text}|{before}|{after}"
    return hashlib.sha256(material.encode("utf-8")).hexdigest()[:16]


def normalize_corrections(
    records: List[Dict[str, Any]],
    source: str = "ChatGPT"
) -> List[Dict[str, Any]]:
    """
    Normalize extracted records into JSONL/Notion-ready records.

    Args:
        records: output list from Cell 06 extract_corrections_via_openai()
        source: logical source label (kept for compatibility if you use it later)

    Returns:
        List of normalized dicts with keys:
          - id, title, skill_id, input, human_correction, human_correction_text,
            model_output, notebook, reason_tags, reason_tags_text, status,
            created_at, cell_tag, source_file, context_snippet, source
    """
    total = len(records or [])
    log_info(f"Normalizing {total} record(s)...")

    now_iso = datetime.now(timezone.utc).isoformat()

    normalized: List[Dict[str, Any]] = []

    for i, r in enumerate(records or [], 1):
        if not isinstance(r, dict):
            log_info(f"Skipping record {i}: not an object")
            continue

        title = _cap(r.get("title", ""), 80)
        if not title:
            log_info(f"Skipping record {i}: missing title")
            continue

        # core fields
        input_text = _cap(r.get("input", ""), TEXT_MAX)
        model_output = _cap(r.get("model_output", ""), TEXT_MAX)

        # human_correction dict
        hc = r.get("human_correction", {}) or {}
        if not isinstance(hc, dict):
            hc = {}

        hc_norm = {
            "before": _cap(hc.get("before", ""), TEXT_MAX),
            "after": _cap(hc.get("after", ""), TEXT_MAX),
            "note": _cap(hc.get("note", ""), NOTE_MAX),
        }

        if not hc_norm["after"]:
            log_info(f"Skipping record {i}: missing human_correction.after")
            continue

        reason_tags = _normalize_tags(r.get("reason_tags", []))
        status = _cap(r.get("status", "raw"), 40).lower() or "raw"

        notebook = _cap(r.get("notebook", NOTEBOOK_NAME), 200)
        cell_tag = _cap(r.get("cell_tag", CELL_TAG), 200)

        source_file = _cap(r.get("source_file", ""), 400)
        context_snippet = _cap(r.get("context_snippet", ""), SNIPPET_MAX)

        # derived helper fields
        hc_text = _make_hc_text(hc_norm)
        reason_tags_text = _cap(", ".join(reason_tags), 500)

        # id + created_at
        rec_id = _hash_id({
            "title": title,
            "input": input_text,
            "human_correction": hc_norm
        })

        item = {
            "id": rec_id,
            "title": _cap(title, TITLE_MAX),

            # Notion-facing fields
            "skill_id": _cap(r.get("skill_id", SKILL_ID), 200),
            "input": input_text,
            "human_correction": hc_norm,
            "human_correction_text": hc_text,
            "model_output": model_output,
            "notebook": notebook,
            "reason_tags": reason_tags,              # list (multi_select向け)
            "reason_tags_text": reason_tags_text,    # fallback
            "status": status,                        # select向け
            "created_at": _cap(r.get("created_at", now_iso), 80),  # date向け
            "cell_tag": cell_tag,

            # Traceability
            "source_file": source_file,
            "context_snippet": context_snippet,

            # Optional bookkeeping
            "source": source,
        }

        normalized.append(item)

    log_info(f"✓ Normalized {len(normalized)} record(s)")

    if normalized:
        s = normalized[0]
        log_info("Sample normalized record:")
        log_info(f"  ID: {s.get('id')}")
        log_info(f"  Skill ID: {s.get('skill_id')}")
        log_info(f"  Title: {s.get('title')}")
        log_info(f"  Status: {s.get('status')}")
        log_info(f"  Source file: {s.get('source_file')}")

    return normalized


log_info("Normalization (v2 records schema) ready ✅")


[INFO] Normalization (v2 records schema) ready ✅


In [9]:
# ============================================================
# Cell 08 — Local JSONL artifact writer
# ============================================================
# Overview:
#   Writes normalized correction records to a local JSONL file for offline inspection.
#   Implements ID-based deduplication to prevent duplicates.
#   Supports append mode for incremental ingestion across multiple runs.
#
# Inputs / Outputs:
#   Input: List of normalized correction dicts (from Cell 07)
#   Output: Appends to skills/human_corrections.jsonl (default)
#
# Notes:
#   - Idempotent: checks existing records by 'id' before writing
#   - Creates parent directory if needed
#   - Logs write statistics (new vs duplicate records)
#   - Each line is a complete JSON object (JSONL format)
#   - Uses conservative file handling (append). Strong concurrency guarantees
#     depend on OS/filesystem and are out of scope for notebook usage.
#

import json
from pathlib import Path
from typing import List, Dict, Any, Set, Optional

# --- Safe defaults (avoid NameError if globals not defined yet) ---
try:
    OUTPUT_JSONL_PATH
except NameError:
    OUTPUT_JSONL_PATH = Path("skills/human_corrections.jsonl")

try:
    DRY_RUN
except NameError:
    DRY_RUN = True


def load_existing_ids(jsonl_path: Path, max_lines: Optional[int] = None) -> Set[str]:
    """
    Load IDs of existing records from JSONL file.

    Args:
        jsonl_path: Path to JSONL artifact file
        max_lines: If provided, read at most this many lines from the end is not supported here;
                   instead it reads from start up to max_lines. (Keep None for full scan.)

    Returns:
        Set of existing record IDs (for deduplication)
    """
    existing_ids: Set[str] = set()

    if not jsonl_path.exists():
        return existing_ids

    try:
        with jsonl_path.open("r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, 1):
                if max_lines is not None and line_num > max_lines:
                    break

                line = line.strip()
                if not line:
                    continue

                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    # Keep going even if file has some broken lines
                    log_info(f"Warning: invalid JSON at line {line_num} (skipped)")
                    continue

                rid = record.get("id")
                if rid:
                    existing_ids.add(str(rid))

        log_info(f"Loaded {len(existing_ids)} existing record ID(s) from {jsonl_path.name}")

    except Exception as e:
        log_error(f"Error reading existing JSONL '{jsonl_path}': {e}")
        # Continue with empty set to allow writes

    return existing_ids


def write_corrections_to_jsonl(
    corrections: List[Dict[str, Any]],
    output_path: Path = OUTPUT_JSONL_PATH,
    dry_run: bool = DRY_RUN,
    verbose_duplicates: bool = False
) -> Dict[str, int]:
    """
    Write normalized corrections to JSONL artifact file.

    Args:
        corrections: List of normalized correction dicts (should include 'id')
        output_path: Target JSONL file path
        dry_run: If True, simulate write without modifying file
        verbose_duplicates: If True, log each skipped duplicate

    Returns:
        Dict with write statistics:
            - 'total': Total corrections processed
            - 'new': Number of new records written
            - 'duplicate': Number of duplicates skipped
            - 'error': Number of write errors
    """
    total = len(corrections or [])
    log_info(f"Writing {total} correction(s) to JSONL...")

    stats = {"total": total, "new": 0, "duplicate": 0, "error": 0}

    if total == 0:
        log_info("No corrections to write")
        return stats

    # --- Load existing IDs for deduplication ---
    existing_ids = load_existing_ids(output_path)

    # --- Filter out duplicates & validate IDs ---
    new_corrections: List[Dict[str, Any]] = []
    for c in corrections:
        rid = str(c.get("id", "")).strip()
        title = str(c.get("title", "untitled")).strip()

        if not rid:
            stats["error"] += 1
            log_error(f"Skipping record with missing 'id': title='{title}'")
            continue

        if rid in existing_ids:
            stats["duplicate"] += 1
            if verbose_duplicates:
                log_info(f"Skipping duplicate: {title} (ID: {rid})")
            continue

        new_corrections.append(c)

    stats["new"] = len(new_corrections)

    if stats["new"] == 0:
        log_info("All corrections already exist in JSONL (duplicates skipped)")
        return stats

    # --- Dry run mode ---
    if dry_run:
        log_info(f"[DRY RUN] Would write {stats['new']} new record(s) to {output_path}")
        for c in new_corrections[:20]:
            log_info(f"  - {c.get('title','untitled')} (ID: {c.get('id')})")
        if len(new_corrections) > 20:
            log_info(f"  ... and {len(new_corrections) - 20} more")
        return stats

    # --- Ensure parent directory exists ---
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # --- Append new records ---
    # Note: append-only is generally safe for typical notebook single-process runs.
    try:
        written = 0
        with output_path.open("a", encoding="utf-8") as f:
            for c in new_corrections:
                try:
                    json_line = json.dumps(c, ensure_ascii=False)
                    f.write(json_line + "\n")
                    written += 1
                except Exception as e:
                    stats["error"] += 1
                    log_error(f"Failed to write record '{c.get('title','untitled')}': {e}")

        stats["new"] = written
        log_info(f"✓ Wrote {written} new record(s) to {output_path}")

    except Exception as e:
        log_error(f"Failed to write JSONL file '{output_path}': {e}")
        # keep stats consistent
        stats["error"] += stats["new"]
        stats["new"] = 0
        raise

    return stats


log_info("JSONL writer ready ✅")
log_info(f"Output path: {OUTPUT_JSONL_PATH}")
log_info(f"Dry run mode: {DRY_RUN}")


[INFO] JSONL writer ready ✅
[INFO] Output path: skills/human_corrections.jsonl
[INFO] Dry run mode: True


In [10]:
# ============================================================
# Cell 09 — Notion database page writer (REST-based, schema-adaptive) [v2]
# ============================================================
# Overview:
#   Writes normalized Skill-ready records (Cell 07 output) to Notion DB.
#   Uses REST API + schema introspection (Cell 04 notion_property_map).
#   Writes ONLY properties that exist in target DB (schema-adaptive).
#
# Inputs / Outputs:
#   Input: List[dict] from Cell 07 normalize_corrections()
#   Output: New pages in Notion database
#
# Notes:
#   - Respects DRY_RUN
#   - Select/multi_select: only writes if option exists (best-effort)
#   - Avoids notion_client SDK differences
#

import os
import time
import requests
from typing import List, Dict, Any, Optional

# --- Safe defaults (avoid NameError due to execution order) ---
try:
    DRY_RUN
except NameError:
    DRY_RUN = True

try:
    NOTION_HC_DB_ID
except NameError:
    NOTION_HC_DB_ID = os.getenv("NOTION_HC_DB_ID", "")

try:
    NOTION_HEADERS
except NameError:
    raise NameError("NOTION_HEADERS is not defined. Define it in Cell 02/03 before running Cell 09.")

# From Cell 04 schema introspection (REST or SDK). Fallbacks provided.
try:
    notion_property_map
except NameError:
    notion_property_map = {}

try:
    notion_title_prop
except NameError:
    notion_title_prop = "Name"  # common default


# -----------------------------
# Notion payload helpers
# -----------------------------
def _rt(text: str):
    text = "" if text is None else str(text)
    return [{"type": "text", "text": {"content": text[:2000]}}]

def _title(text: str):
    text = "" if text is None else str(text)
    return [{"type": "text", "text": {"content": text[:2000]}}]

def notion_create_page(database_id: str, properties: Dict[str, Any]) -> requests.Response:
    payload = {"parent": {"database_id": database_id}, "properties": properties}
    return requests.post(
        "https://api.notion.com/v1/pages",
        headers=NOTION_HEADERS,
        json=payload,
        timeout=30
    )

def _get_prop_type(prop_name: str) -> Optional[str]:
    cfg = notion_property_map.get(prop_name, {})
    return cfg.get("type") if isinstance(cfg, dict) else None

def _get_select_options(prop_name: str) -> List[str]:
    """
    Extract select/multi_select option names from notion_property_map[prop]['config'].
    """
    cfg = notion_property_map.get(prop_name, {}).get("config", {})
    if not isinstance(cfg, dict):
        return []
    ptype = cfg.get("type")

    if ptype == "select":
        return [o.get("name") for o in cfg.get("select", {}).get("options", []) if o.get("name")]
    if ptype == "multi_select":
        return [o.get("name") for o in cfg.get("multi_select", {}).get("options", []) if o.get("name")]
    return []


# -----------------------------
# Schema-adaptive mapping
# -----------------------------
def build_notion_properties_schema_adaptive(c: Dict[str, Any]) -> Dict[str, Any]:
    """
    Build Notion properties payload using best-effort mapping from normalized keys (Cell 07)
    to your Notion DB columns. Writes only existing properties.
    """
    props: Dict[str, Any] = {}

    # --- 1) Title property (required in Notion DB) ---
    title_value = c.get("title", "") or "untitled"
    if notion_title_prop in notion_property_map and _get_prop_type(notion_title_prop) == "title":
        props[notion_title_prop] = {"title": _title(title_value)}
    else:
        # Try any title-type property from schema
        title_candidates = [k for k, v in notion_property_map.items() if isinstance(v, dict) and v.get("type") == "title"]
        if title_candidates:
            props[title_candidates[0]] = {"title": _title(title_value)}
        else:
            raise ValueError("No title property detected in Notion DB schema.")

    # --- 2) Preferred mapping for YOUR current DB (from screenshot) ---
    # normalized_key -> candidate Notion property names (in priority order)
    field_map = {
        "skill_id": ["Skill ID", "Skill_ID", "SkillId"],
        "input": ["Input"],
        "human_correction_text": ["Human Correction", "Human Correction Text", "Human_Correction"],
        "model_output": ["Model Output", "Model_Output"],
        "notebook": ["Notebook"],
        "reason_tags": ["Reason Tags", "Reason_Tags"],
        "status": ["Status"],
        "created_at": ["Created At", "Created_At", "Timestamp", "Date"],
        "cell_tag": ["Cell Tag", "Cell_Tag"],
        "source_file": ["Source File", "Source_File", "File"],
        "id": ["ID", "Record ID", "Record_ID", "Hash_ID"],
        "context_snippet": ["Context Snippet", "Context", "Snippet"],
    }

    def _set_if_exists(prop_name: str, payload: Dict[str, Any]):
        if prop_name in notion_property_map:
            props[prop_name] = payload

    # --- rich_text writes ---
    for nk in ["skill_id", "input", "human_correction_text", "model_output", "notebook", "cell_tag", "source_file", "id", "context_snippet"]:
        val = c.get(nk, "")
        for candidate in field_map.get(nk, []):
            if candidate in notion_property_map and _get_prop_type(candidate) == "rich_text":
                _set_if_exists(candidate, {"rich_text": _rt(val)})
                break

    # --- date write (created_at) ---
    dt = c.get("created_at") or ""
    if dt:
        for candidate in field_map.get("created_at", []):
            if candidate in notion_property_map and _get_prop_type(candidate) == "date":
                _set_if_exists(candidate, {"date": {"start": str(dt)}})
                break

    # --- select write (status) with fallback ---
    status_val = str(c.get("status", "") or "").strip()
    if status_val:
        # select first
        for candidate in field_map.get("status", []):
            if candidate in notion_property_map and _get_prop_type(candidate) == "select":
                options = _get_select_options(candidate)
                if options and status_val not in options:
                    # option not present -> don't write select to avoid failure
                    break
                _set_if_exists(candidate, {"select": {"name": status_val}})
                break
        else:
            # fallback to rich_text if exists
            for candidate in field_map.get("status", []):
                if candidate in notion_property_map and _get_prop_type(candidate) == "rich_text":
                    _set_if_exists(candidate, {"rich_text": _rt(status_val)})
                    break

    # --- multi_select write (reason_tags) with fallback ---
    tags = c.get("reason_tags", [])
    if isinstance(tags, str):
        tags = [t.strip() for t in tags.split(",") if t.strip()]
    if isinstance(tags, list) and tags:
        # multi_select first
        wrote = False
        for candidate in field_map.get("reason_tags", []):
            if candidate in notion_property_map and _get_prop_type(candidate) == "multi_select":
                options = _get_select_options(candidate)
                # only keep allowed options if options exist
                if options:
                    tags_use = [t for t in tags if t in options]
                else:
                    tags_use = tags
                if tags_use:
                    _set_if_exists(candidate, {"multi_select": [{"name": t} for t in tags_use]})
                    wrote = True
                break

        # fallback: write comma string into rich_text if multi_select not possible
        if not wrote:
            tags_text = ", ".join([str(t) for t in tags])
            for candidate in field_map.get("reason_tags", []):
                if candidate in notion_property_map and _get_prop_type(candidate) == "rich_text":
                    _set_if_exists(candidate, {"rich_text": _rt(tags_text)})
                    break

    return props


def write_corrections_to_notion(
    corrections: List[Dict[str, Any]],
    database_id: Optional[str] = None,
    dry_run: bool = DRY_RUN,
    max_retries: int = 3
) -> Dict[str, int]:
    """
    Write normalized records to Notion database as new pages.

    Returns:
      {'total': N, 'success': x, 'error': y}
    """
    db_id = database_id or NOTION_HC_DB_ID
    if not db_id:
        raise ValueError("database_id is empty. Set NOTION_HC_DB_ID in env.txt")

    total = len(corrections or [])
    log_info(f"Writing {total} record(s) to Notion...")

    stats = {"total": total, "success": 0, "error": 0}

    if total == 0:
        log_info("No records to write")
        return stats

    if dry_run:
        log_info(f"[DRY RUN] Would create {total} Notion page(s)")
        for c in corrections[:20]:
            log_info(f"  - {c.get('title','untitled')} (status={c.get('status','')}, id={c.get('id','')})")
        if total > 20:
            log_info(f"  ... and {total - 20} more")
        stats["success"] = total
        return stats

    for i, c in enumerate(corrections, 1):
        title = c.get("title", "untitled")
        log_info(f"Writing {i}/{total}: {title}")

        try:
            props = build_notion_properties_schema_adaptive(c)
        except Exception as e:
            log_error(f"Failed to build properties for '{title}': {e}")
            stats["error"] += 1
            continue

        success = False
        for attempt in range(1, max_retries + 1):
            try:
                r = notion_create_page(db_id, props)

                if r.status_code in (200, 201):
                    data = r.json()
                    page_id = data.get("id", "")
                    log_info(f"✓ Created page: {page_id[:8]}...{page_id[-4:]}" if page_id else "✓ Created page")
                    stats["success"] += 1
                    success = True
                    break

                if r.status_code == 429 and attempt < max_retries:
                    wait_time = 2 ** attempt
                    log_info(f"Rate limited (429). Waiting {wait_time}s then retry...")
                    time.sleep(wait_time)
                    continue

                log_error(f"Notion API error for '{title}': HTTP {r.status_code} - {r.text}")
                break

            except Exception as e:
                log_error(f"Attempt {attempt}/{max_retries} failed for '{title}': {type(e).__name__}: {e}")
                if attempt < max_retries:
                    time.sleep(2 ** attempt)

        if not success:
            stats["error"] += 1
            log_error(f"Failed to create page after {max_retries} attempts: {title}")

    log_info(f"Notion write complete: {stats['success']} success, {stats['error']} errors")
    return stats


log_info("Notion writer (v2 schema) ready ✅")
log_info(f"Target database: {NOTION_HC_DB_ID[:8]}...{NOTION_HC_DB_ID[-4:]}" if NOTION_HC_DB_ID else "Target database: (unset)")
log_info(f"Dry run mode: {DRY_RUN}")


[INFO] Notion writer (v2 schema) ready ✅
[INFO] Target database: 2ee8e0e4...6cdf
[INFO] Dry run mode: True


In [11]:
# ============================================================
# Cell 10 — End-to-end pipeline orchestrator (v2: records schema) [FINAL]
# ============================================================
# Overview:
#   Orchestrates: file load → OpenAI extraction (records schema) → normalization (v2) →
#                JSONL write → Notion write.
#   Compatible with:
#     - Cell 06 (records schema)
#     - Cell 07 normalize_corrections(records, source=...)
#     - Cell 08 JSONL writer (expects 'id')
#     - Cell 09 Notion writer (v2 mapping; writes created_at -> Created At)
#
# Notes:
#   - Continues processing on file-level failures
#   - JSONL dedup is the source of truth for "new vs duplicate"
#   - Ensures created_at exists (UTC ISO) before writes
#

from typing import Union, List, Dict, Any
from pathlib import Path
from datetime import datetime, timezone

def _ensure_created_at(records: List[Dict[str, Any]]) -> None:
    """Ensure each record has created_at (UTC ISO). Mutates in place."""
    now_iso = datetime.now(timezone.utc).isoformat()
    for r in records:
        if not r.get("created_at"):
            r["created_at"] = now_iso

def _mask_id(s: str) -> str:
    if not s:
        return ""
    return f"{s[:8]}...{s[-4:]}" if len(s) > 12 else s


def process_single_file(
    file_path: Path,
    dry_run: bool = DRY_RUN
) -> Dict[str, Any]:
    """
    Process a single chat log file end-to-end and return a result dict.
    (This function is used by run_pipeline; do not keep alternative logic elsewhere.)
    """
    result = {
        "file": file_path.name,
        "success": False,
        "corrections_extracted": 0,
        "corrections_normalized": 0,
        "jsonl_stats": {},
        "notion_stats": {},
        "error": None,
    }

    try:
        log_info(f"\n{'='*60}")
        log_info(f"Processing: {file_path.name}")
        log_info(f"{'='*60}")

        # 1) Load
        chat_log = load_chat_log(file_path)

        # 2) Extract (records schema)
        raw_records = extract_corrections_via_openai(chat_log)
        result["corrections_extracted"] = len(raw_records)

        if not raw_records:
            log_info("No records extracted; treating as success (no-op)")
            result["success"] = True
            return result

        # 3) Normalize (v2)
        normalized = normalize_corrections(raw_records, source="ChatGPT")
        result["corrections_normalized"] = len(normalized)

        if not normalized:
            log_info("No records survived normalization; treating as success (no-op)")
            result["success"] = True
            return result

        # 3.5) Ensure created_at exists (UTC ISO)
        _ensure_created_at(normalized)

        # Debug: show created_at presence (first record only)
        try:
            log_info(f"[DEBUG] created_at (first): {normalized[0].get('created_at')}")
            log_info(f"[DEBUG] keys (first): {list(normalized[0].keys())}")
        except Exception:
            pass

        # 4) Dedup baseline BEFORE JSONL write
        existing_before = load_existing_ids(OUTPUT_JSONL_PATH)

        # 5) Write JSONL (dedup + append)
        jsonl_stats = write_corrections_to_jsonl(normalized, dry_run=dry_run)
        result["jsonl_stats"] = jsonl_stats

        # 6) Compute new_records based on "existing_before"
        new_records = [r for r in normalized if r.get("id") and r["id"] not in existing_before]

        # Debug: show Created At going into Notion
        if new_records:
            log_info(f"[DEBUG] new_records: {len(new_records)}")
            log_info(f"[DEBUG] Notion first new created_at: {new_records[0].get('created_at')} (id={_mask_id(new_records[0].get('id',''))})")
        else:
            log_info("[DEBUG] new_records: 0 (all duplicates or missing id)")

        # 7) Write to Notion (only new_records)
        notion_stats = write_corrections_to_notion(
            new_records,
            dry_run=dry_run
        )
        result["notion_stats"] = notion_stats

        result["success"] = True
        log_info(f"✓ File processing complete: {file_path.name}")

    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"
        result["success"] = False
        log_error(f"Failed to process {file_path.name}: {result['error']}")

    return result


def run_pipeline(
    input_path: Union[str, Path, List[Path]],
    dry_run: bool = DRY_RUN,
    max_records: int = None
) -> Dict[str, Any]:
    """
    Run pipeline on:
      - a single file,
      - a folder (discover *.txt),
      - or a list of files.
    """
    input_path = Path(input_path) if isinstance(input_path, str) else input_path

    if isinstance(input_path, list):
        files = input_path
    elif input_path.is_file():
        files = [input_path]
    elif input_path.is_dir():
        files = discover_chat_logs(input_path)
    else:
        raise ValueError(f"Invalid input_path: {input_path}")

    if max_records is not None:
        files = files[:max_records]
        log_info(f"Limited to first {max_records} file(s)")

    if len(files) == 0:
        log_info("No files to process")
        return {
            "files_processed": 0,
            "files_success": 0,
            "files_failed": 0,
            "total_corrections_extracted": 0,
            "total_corrections_normalized": 0,
            "total_jsonl_new": 0,
            "total_notion_success": 0,
            "total_notion_errors": 0,
            "file_results": [],
            "dry_run": dry_run
        }

    summary = {
        "files_processed": len(files),
        "files_success": 0,
        "files_failed": 0,
        "total_corrections_extracted": 0,
        "total_corrections_normalized": 0,
        "total_jsonl_new": 0,
        "total_notion_success": 0,
        "total_notion_errors": 0,
        "file_results": [],
        "dry_run": dry_run
    }

    log_info(f"\n{'='*60}")
    log_info(f"PIPELINE START: {len(files)} file(s) to process")
    log_info(f"Dry run: {dry_run}")
    log_info(f"{'='*60}\n")

    for i, file_path in enumerate(files, 1):
        log_info(f"\n[{i}/{len(files)}] Processing: {file_path.name}")
        res = process_single_file(file_path, dry_run=dry_run)
        summary["file_results"].append(res)

        if res["success"]:
            summary["files_success"] += 1
        else:
            summary["files_failed"] += 1

        summary["total_corrections_extracted"] += res.get("corrections_extracted", 0)
        summary["total_corrections_normalized"] += res.get("corrections_normalized", 0)
        summary["total_jsonl_new"] += res.get("jsonl_stats", {}).get("new", 0)
        summary["total_notion_success"] += res.get("notion_stats", {}).get("success", 0)
        summary["total_notion_errors"] += res.get("notion_stats", {}).get("error", 0)

    log_info(f"\n{'='*60}")
    log_info("PIPELINE COMPLETE")
    log_info(f"{'='*60}")
    log_info(f"Files processed: {summary['files_processed']}")
    log_info(f"  Success: {summary['files_success']}")
    log_info(f"  Failed: {summary['files_failed']}")
    log_info(f"Corrections extracted: {summary['total_corrections_extracted']}")
    log_info(f"Corrections normalized: {summary['total_corrections_normalized']}")
    log_info(f"JSONL records written (new): {summary['total_jsonl_new']}")
    log_info(f"Notion pages created: {summary['total_notion_success']}")
    log_info(f"Notion write errors: {summary['total_notion_errors']}")

    if summary["files_failed"] > 0:
        log_info("\nFailed files:")
        for r in summary["file_results"]:
            if not r["success"]:
                log_info(f"  - {r['file']}: {r['error']}")

    log_info(f"\nOutput JSONL: {OUTPUT_JSONL_PATH}")
    log_info(f"Notion database: {notion_hc_db_id[:8]}...{notion_hc_db_id[-4:]}" if notion_hc_db_id else "Notion database: (unset)")
    log_info(f"{'='*60}\n")

    return summary


log_info("Pipeline orchestrator (v2 records schema) ready ✅")
log_info("Usage: run_pipeline(INPUT_FOLDER) or run_pipeline(Path('data/chat_logs/example.txt'))")


# ============================================================
# Optional: Widget UI (file picker upload)
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output

UPLOAD_DIR = Path("data/chat_logs/_uploaded")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

uploader = widgets.FileUpload(
    accept=".txt",
    multiple=True,
    description="Choose .txt files"
)

dry_run_toggle = widgets.ToggleButtons(
    options=[("DRY_RUN=True", True), ("DRY_RUN=False", False)],
    value=True,
    description="Mode:"
)

run_btn = widgets.Button(
    description="Save & Run pipeline",
    button_style="success"
)

out = widgets.Output()

def _save_uploaded_files() -> List[Path]:
    saved_paths: List[Path] = []
    if not uploader.value:
        return saved_paths

    val = uploader.value
    if isinstance(val, dict):
        items = [{"name": name, "content": meta.get("content")} for name, meta in val.items()]
    else:
        items = list(val)

    for item in items:
        name = item.get("name") or "uploaded.txt"
        content = item.get("content")
        if content is None:
            continue

        target = UPLOAD_DIR / name
        if target.exists():
            stem, suffix = target.stem, target.suffix
            k = 1
            while True:
                candidate = UPLOAD_DIR / f"{stem}__{k}{suffix}"
                if not candidate.exists():
                    target = candidate
                    break
                k += 1

        with open(target, "wb") as f:
            f.write(content)

        saved_paths.append(target)

    return saved_paths

def on_run_clicked(_):
    with out:
        clear_output()
        dry_run = dry_run_toggle.value

        saved = _save_uploaded_files()
        if not saved:
            print("No files uploaded. Click 'Choose .txt files' first.")
            return

        print(f"Saved {len(saved)} file(s) to: {UPLOAD_DIR}")
        for p in saved:
            print(f" - {p}")

        print("-" * 60)
        summary = run_pipeline(saved, dry_run=dry_run, max_records=None)
        print("\nDone.")
        print(summary)

run_btn.on_click(on_run_clicked)

display(widgets.VBox([
    uploader,
    dry_run_toggle,
    run_btn,
    out
]))


[INFO] Pipeline orchestrator (v2 records schema) ready ✅
[INFO] Usage: run_pipeline(INPUT_FOLDER) or run_pipeline(Path('data/chat_logs/example.txt'))


In [12]:
# ============================================================
# Cell 11 — Example invocation and testing
# ============================================================
# Overview:
#   Demonstrates pipeline usage with example invocations.
#   Provides test scenarios for single-file, batch, and dry-run modes.
#   Safe to run (defaults to DRY_RUN=True); shows expected workflow.
#
# Inputs / Outputs:
#   Reads: Uses configuration from Cells 01-10
#   Outputs: Executes pipeline and displays summary statistics
#
# Notes:
#   - Change DRY_RUN to False in Cell 01 to enable actual writes
#   - Requires valid chat log files in INPUT_FOLDER
#   - Demonstrates error handling and partial failure scenarios
#   - Safe to re-run (JSONL deduplication prevents duplicates)
#

# --- Example 1: Process all files in default input folder (dry run) ---
log_info("\n" + "="*60)
log_info("EXAMPLE 1: Batch processing with dry run")
log_info("="*60)

# Discover available files
if INPUT_FOLDER.exists():
    available_files = discover_chat_logs(INPUT_FOLDER)
    
    if available_files:
        log_info(f"Found {len(available_files)} file(s) in {INPUT_FOLDER}")
        
        # Run pipeline on all files (respects DRY_RUN and MAX_RECORDS from Cell 01)
        summary = run_pipeline(
            input_path=INPUT_FOLDER,
            dry_run=DRY_RUN,
            max_records=MAX_RECORDS
        )
        
        # Display summary
        log_info("\nPipeline Summary:")
        log_info(f"  Files processed: {summary['files_processed']}")
        log_info(f"  Success rate: {summary['files_success']}/{summary['files_processed']}")
        log_info(f"  Total corrections extracted: {summary['total_corrections_extracted']}")
        log_info(f"  JSONL new records: {summary['total_jsonl_new']}")
        log_info(f"  Notion pages created: {summary['total_notion_success']}")
        
    else:
        log_info(f"No chat log files found in {INPUT_FOLDER}")
        log_info("To test the pipeline:")
        log_info("  1. Create a .txt file in data/chat_logs/")
        log_info("  2. Add content with format:")
        log_info("     Original Prompt:\\n<your prompt>\\n\\nConversation:\\n<chat log>")
        log_info("  3. Re-run this cell")
else:
    log_info(f"Input folder not found: {INPUT_FOLDER}")
    log_info("Creating folder for future use...")
    INPUT_FOLDER.mkdir(parents=True, exist_ok=True)
    log_info(f"✓ Created {INPUT_FOLDER}")
    log_info("Add .txt files with chat logs to test the pipeline")


# --- Example 2: Process a single file (uncomment to test) ---
# log_info("\n" + "="*60)
# log_info("EXAMPLE 2: Single file processing")
# log_info("="*60)
# 
# single_file = INPUT_FOLDER / 'example_chat_log.txt'
# if single_file.exists():
#     result = process_single_file(single_file, dry_run=True)
#     
#     log_info(f"\nResult for {result['file']}:")
#     log_info(f"  Success: {result['success']}")
#     log_info(f"  Corrections extracted: {result['corrections_extracted']}")
#     log_info(f"  Corrections normalized: {result['corrections_normalized']}")
#     
#     if result['error']:
#         log_error(f"  Error: {result['error']}")
# else:
#     log_info(f"Example file not found: {single_file}")


# --- Example 3: Inspect JSONL output (if exists) ---
log_info("\n" + "="*60)
log_info("EXAMPLE 3: JSONL artifact inspection")
log_info("="*60)

if OUTPUT_JSONL_PATH.exists():
    existing_ids = load_existing_ids(OUTPUT_JSONL_PATH)
    log_info(f"JSONL artifact exists: {OUTPUT_JSONL_PATH}")
    log_info(f"Total records: {len(existing_ids)}")
    
    # Show sample record
    with OUTPUT_JSONL_PATH.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                sample_record = json.loads(line)
                log_info("\nSample record:")
                log_info(f"  ID: {sample_record.get('id', 'N/A')}")
                log_info(f"  Title: {sample_record.get('title', 'N/A')}")
                log_info(f"  Type: {sample_record.get('correction_type', 'N/A')}")
                log_info(f"  Source: {sample_record.get('source_file', 'N/A')}")
                log_info(f"  Timestamp: {sample_record.get('timestamp', 'N/A')}")
                break
else:
    log_info(f"JSONL artifact not yet created: {OUTPUT_JSONL_PATH}")
    log_info("Will be created on first successful pipeline run")


# --- Example 4: Production mode instructions ---
log_info("\n" + "="*60)
log_info("EXAMPLE 4: Production mode (actual writes)")
log_info("="*60)

if DRY_RUN:
    log_info("Currently in DRY RUN mode (no actual writes)")
    log_info("\nTo enable production mode:")
    log_info("  1. Go to Cell 01")
    log_info("  2. Set: DRY_RUN = False")
    log_info("  3. Re-run all cells (or at least Cells 01, 08, 09, 10)")
    log_info("  4. Re-run this cell to execute actual writes")
    log_info("\nProduction run example:")
    log_info("  summary = run_pipeline(INPUT_FOLDER, dry_run=False, max_records=5)")
else:
    log_info("✓ Production mode ENABLED (writes will be executed)")
    log_info("Pipeline will create actual Notion pages and JSONL records")
    log_info("\nTo test production mode safely:")
    log_info("  summary = run_pipeline(INPUT_FOLDER, dry_run=False, max_records=1)")


# --- Display current configuration ---
log_info("\n" + "="*60)
log_info("Current Configuration")
log_info("="*60)
log_info(f"LLM: {llm_provider}/{llm_model} @ temp={llm_temperature}")
log_info(f"Dry run: {DRY_RUN}")
log_info(f"Max records: {MAX_RECORDS if MAX_RECORDS else 'unlimited'}")
log_info(f"Input folder: {INPUT_FOLDER}")
log_info(f"Output JSONL: {OUTPUT_JSONL_PATH}")
log_info(f"Notion DB: {notion_hc_db_id[:8]}...{notion_hc_db_id[-4:]}")
log_info("="*60)

log_info("\n✓ Example cell ready - pipeline is operational")
log_info("Modify examples above to test specific scenarios")


[INFO] 
[INFO] EXAMPLE 1: Batch processing with dry run
[INFO] ============================================================
[INFO] Found 0 chat log file(s) in data/chat_logs
[INFO] No chat log files found in data/chat_logs
[INFO] To test the pipeline:
[INFO]   1. Create a .txt file in data/chat_logs/
[INFO]   2. Add content with format:
[INFO]      Original Prompt:\n<your prompt>\n\nConversation:\n<chat log>
[INFO]   3. Re-run this cell
[INFO] 
[INFO] EXAMPLE 3: JSONL artifact inspection
[INFO] ============================================================
[INFO] Loaded 40 existing record ID(s) from human_corrections.jsonl
[INFO] JSONL artifact exists: skills/human_corrections.jsonl
[INFO] Total records: 40
[INFO] 
Sample record:
[INFO]   ID: e96156e193c26714
[INFO]   Title: Switch to Notion REST API for Authentication Validation
[INFO]   Type: Structure
[INFO]   Source: 20260122 test__3.txt
[INFO]   Timestamp: 2026-01-22T01:54:44.439146+00:00
[INFO] 
[INFO] EXAMPLE 4: Production mode (a